In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Register your exact Encoder_Module path
ENCODER_DIR = r"E:\AVIA\src\Enocoder_Module"
if ENCODER_DIR not in sys.path:
    sys.path.append(ENCODER_DIR)

# Import the 4 standalone encoders you saved
from cardiac_encoder import AVIACardiacRespEncoder, RealtimeCardiacInference, BIDMC_FEATURE_COLS
import importlib

stress_encoder = importlib.import_module("stress_encoder")

RealtimeStressInference = stress_encoder.RealtimeStressInference

# Support the class name currently implemented in stress_encoder.py.
AVIAStressEncoder = getattr(
    stress_encoder,
    "AVIAStressEncoder",
    getattr(stress_encoder, "StressEncoder", None)
)

if AVIAStressEncoder is None:
    raise ImportError(
        "stress_encoder.py must define AVIAStressEncoder or StressEncoder."
    )

WESAD_FEATURE_COLS = getattr(
    stress_encoder,
    "WESAD_FEATURE_COLS",
    [f"stress_{i}" for i in range(32)]
)
from sleep_encoder import AVIASleepEncoder, RealtimeSleepInference, SLEEP_FEATURE_COLS
from nasa_context_encoder import AVIANASAContextEncoder

# 2. Define project data & model directories
BASE_DIR      = r"E:\AVIA"
BIDMC_DIR     = os.path.join(BASE_DIR, "data", "processed", "bidmc")
WESAD_DIR     = os.path.join(BASE_DIR, "data", "processed", "wesad")
SLEEP_DIR     = os.path.join(BASE_DIR, "data", "processed", "sleep_edf")
OSD919_DIR    = os.path.join(BASE_DIR, "data", "processed", "osd919")
OSD569_DIR    = os.path.join(BASE_DIR, "data", "raw", "NASA_OSD569")
MODELS_DIR    = os.path.join(BASE_DIR, "models")
FIGURES_DIR   = os.path.join(BASE_DIR, "reports", "figures")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Successfully imported all 4 encoders from: {ENCODER_DIR}")
print(f"[+] Active Compute Device: {device}")

ImportError: cannot import name 'AVIAStressEncoder' from 'stress_encoder' (E:\AVIA\src\Enocoder_Module\stress_encoder.py)

In [3]:
# 1. Load the 32-dim wearable embeddings generated in Steps 2 & 3
df_card_e   = pd.read_csv(os.path.join(BIDMC_DIR, "bidmc_32dim_embeddings.csv"))
df_stress_e = pd.read_csv(os.path.join(WESAD_DIR, "wesad_32dim_embeddings.csv"))
df_sleep_e  = pd.read_csv(os.path.join(SLEEP_DIR, "sleep_edf_32dim_embeddings.csv"))

card_cols   = [f"cardiac_emb_{i}" for i in range(32)]
stress_cols = [f"stress_emb_{i}" for i in range(32)]
sleep_cols  = [f"sleep_emb_{i}" for i in range(32)]

E_card   = df_card_e[card_cols].values.astype(np.float32)
P_card   = df_card_e["cardio_resp_anomaly_prob"].values.astype(np.float32)

E_stress = df_stress_e[stress_cols].values.astype(np.float32)
P_stress = df_stress_e["stress_probability"].values.astype(np.float32)

E_sleep  = df_sleep_e[sleep_cols].values.astype(np.float32)
S_stage  = df_sleep_e["pred_stage_id"].values.astype(int)

# Map AASM sleep stages to physiological recovery weights (N3 Deep=0.95, REM=0.85, N2=0.70, N1=0.40, Wake=0.15)
stage_to_rec = {3: 0.95, 4: 0.85, 2: 0.70, 1: 0.40, 0: 0.15}
R_sleep = np.array([stage_to_rec.get(s, 0.5) for s in S_stage], dtype=np.float32)

# 2. Load NASA OSD-569 (Inspiration4 Human CBC) & OSD-919 (Validated Organ Deltas)
cbc_path    = os.path.join(OSD569_DIR, "LSDS-7_Complete_Blood_Count_CBC_TRANSFORMED.csv")
osd919_path = os.path.join(OSD919_DIR, "osd919_29_tissues_validated_stats.csv")

df_cbc = pd.read_csv(cbc_path)
df_cbc.columns = [c.strip().replace("\ufeff", "") for c in df_cbc.columns]
df_cbc["Crew_ID"]   = df_cbc["Sample Name"].apply(lambda x: x.split("_")[0])
df_cbc["Timepoint"] = df_cbc["Sample Name"].apply(lambda x: x.split("_")[2])

cbc_cols = [
    "white_blood_cell_count_value_thousand_per_microliter",
    "red_blood_cell_count_value_million_per_microliter",
    "hemoglobin_value_percent", "hematocrit_value_percent",
    "platelet_count_value_thousand_per_microliter",
    "absolute_neutrophils_value_cells_per_microliter",
    "absolute_lymphocytes_value_cells_per_microliter",
    "absolute_monocytes_value_cells_per_microliter",
    "absolute_eosinophils_value_cells_per_microliter",
    "absolute_basophils_value_cells_per_microliter",
    "mcv_value_femtoliter", "mch_value_picogram",
    "mchc_value_grams_per_deciliter", "rdw_value_percent", "mpv_value_femtoliter"
]

# Impute the 7 NaN entries in C002's MCV/MCH/MCHC using per-crew median so no NaNs enter the fusion network
for c in cbc_cols:
    df_cbc[c] = pd.to_numeric(df_cbc[c], errors="coerce")
    df_cbc[c] = df_cbc.groupby("Crew_ID")[c].transform(lambda s: s.fillna(s.median()))
    df_cbc[c] = df_cbc[c].fillna(df_cbc[c].median())

df_osd919 = pd.read_csv(osd919_path)
key_organs = ["Bone", "Skeletal_Stem_Cells", "Aorta", "Heart_Atria", "Blood", "Spleen", "Kidney", "Skin"]
sub_919 = df_osd919.set_index("tissue").reindex(key_organs).fillna(0.0)
osd919_16d = np.concatenate([sub_919["uncoupling_delta"].values, sub_919["inflammatory_delta"].values]).astype(np.float32)

context_vectors = {}
for _, row in df_cbc.iterrows():
    cid, tp = row["Crew_ID"], row["Timepoint"]
    base_row = df_cbc[(df_cbc["Crew_ID"] == cid) & (df_cbc["Timepoint"] == "L-3")].iloc[0]
    shifts = [(float(row[c]) - float(base_row[c])) / (abs(float(base_row[c])) + 1e-5) for c in cbc_cols]
    
    b_nlr = float(base_row["absolute_neutrophils_value_cells_per_microliter"]) / max(1.0, float(base_row["absolute_lymphocytes_value_cells_per_microliter"]))
    c_nlr = float(row["absolute_neutrophils_value_cells_per_microliter"]) / max(1.0, float(row["absolute_lymphocytes_value_cells_per_microliter"]))
    nlr_s = (c_nlr - b_nlr) / (b_nlr + 1e-5)
    
    is_flight = 1.0 if tp in ["R+1", "R+45"] else 0.2
    z_ctx = np.concatenate([np.array(shifts + [nlr_s], dtype=np.float32), osd919_16d * is_flight])
    context_vectors[f"{cid}_{tp}"] = (np.nan_to_num(z_ctx, nan=0.0), float(np.nan_to_num(abs(nlr_s), nan=0.0)))

print(f"[+] Loaded Embeddings -> Cardiac: {E_card.shape}, Stress: {E_stress.shape}, Sleep: {E_sleep.shape}, NASA Contexts: {len(context_vectors)}")

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\AVIA\\data\\processed\\osd919\\osd919_29_tissues_validated_stats.csv'

In [ ]:
class MultimodalFusionModule(nn.Module):
    """
    Intermediate Representation Fusion Layer:
    Combines [e_cardiac (32) || e_stress (32) || e_sleep (32) || z_context (32)] -> 128-dim Human State Vector.
    Handles missing sensor modalities explicitly via a binary availability mask.
    """
    def __init__(self, in_dim: int = 128, state_dim: int = 128):
        super().__init__()
        self.modality_gates = nn.Parameter(torch.ones(4)) # Learnable modality attention weights
        self.fusion_mlp = nn.Sequential(
            nn.Linear(in_dim, state_dim),
            nn.LayerNorm(state_dim),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(state_dim, state_dim),
            nn.LayerNorm(state_dim),
            nn.SiLU()
        )

    def forward(self, e_card, e_stress, e_sleep, z_ctx, modality_mask=None):
        # Explicitly zero out any missing/offline sensor modality and rescale active gates
        if modality_mask is not None:
            e_card   = e_card * modality_mask[:, 0:1]
            e_stress = e_stress * modality_mask[:, 1:2]
            e_sleep  = e_sleep * modality_mask[:, 2:3]
            z_ctx    = z_ctx * modality_mask[:, 3:4]

        w = torch.softmax(self.modality_gates, dim=0) * 4.0
        concat_128 = torch.cat([e_card * w[0], e_stress * w[1], e_sleep * w[2], z_ctx * w[3]], dim=-1)
        fused_state = self.fusion_mlp(concat_128) + concat_128 # Residual preserves modality identity
        return fused_state


class TemporalBaselineEngine(nn.Module):
    """
    Models 24h-72h physiological trajectories via a 2-layer GRU on fused 128-dim state representations.
    Outputs:
      1. Anomaly Score (0.0 to 1.0)
      2. Sleep / Recovery Profile Score (0.0 to 1.0)
      3. Autonomic Stress Profile Score (0.0 to 1.0)
    """
    def __init__(self, state_dim: int = 128, hidden_dim: int = 64):
        super().__init__()
        self.fusion = MultimodalFusionModule(128, state_dim)
        self.gru = nn.GRU(
            input_size=state_dim, hidden_size=hidden_dim,
            num_layers=2, batch_first=True, dropout=0.1
        )
        self.anomaly_head  = nn.Sequential(nn.Linear(hidden_dim, 32), nn.SiLU(), nn.Linear(32, 1), nn.Sigmoid())
        self.recovery_head = nn.Sequential(nn.Linear(hidden_dim, 32), nn.SiLU(), nn.Linear(32, 1), nn.Sigmoid())
        self.stress_head   = nn.Sequential(nn.Linear(hidden_dim, 32), nn.SiLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x_seq, mask_seq=None):
        B, T, _ = x_seq.shape
        flat_x = x_seq.view(B * T, 128)
        e_c, e_st, e_sl, z_cx = flat_x[:, :32], flat_x[:, 32:64], flat_x[:, 64:96], flat_x[:, 96:128]
        flat_mask = mask_seq.view(B * T, 4) if mask_seq is not None else None

        fused_flat = self.fusion(e_c, e_st, e_sl, z_cx, flat_mask)
        fused_seq  = fused_flat.view(B, T, 128)

        gru_out, _ = self.gru(fused_seq)
        anom_score = self.anomaly_head(gru_out).squeeze(-1)
        rec_score  = self.recovery_head(gru_out).squeeze(-1)
        str_score  = self.stress_head(gru_out).squeeze(-1)
        return anom_score, rec_score, str_score, fused_seq

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

card_norm_idx   = np.where(P_card < 0.25)[0]
card_anom_idx   = np.where(P_card > 0.75)[0]
stress_norm_idx = np.where(P_stress < 0.25)[0]
stress_elev_idx = np.where(P_stress > 0.75)[0]
sleep_good_idx  = np.where(np.isin(S_stage, [2, 3, 4]))[0]
sleep_poor_idx  = np.where(np.isin(S_stage, [0, 1]))[0]

ctx_keys = list(context_vectors.keys())
pre_keys = [k for k in ctx_keys if "L-" in k]
flt_keys = [k for k in ctx_keys if "R+1" in k or "R+45" in k]

N_SEQ   = 600
SEQ_LEN = 6  # 6 steps across 72 hours (0h, 12h, 24h, 36h, 48h, 72h)

X_seq  = np.zeros((N_SEQ, SEQ_LEN, 128), dtype=np.float32)
Y_anom = np.zeros((N_SEQ, SEQ_LEN), dtype=np.float32)
Y_rec  = np.zeros((N_SEQ, SEQ_LEN), dtype=np.float32)
Y_str  = np.zeros((N_SEQ, SEQ_LEN), dtype=np.float32)

for i in range(N_SEQ):
    is_progressive_dev = (i % 2 == 1)
    ctx_k = np.random.choice(flt_keys if is_progressive_dev else pre_keys)
    z_ctx, ctx_risk = context_vectors[ctx_k]

    for t in range(SEQ_LEN):
        dev_prob = (t / (SEQ_LEN - 1)) if is_progressive_dev else 0.05
        c_idx  = np.random.choice(card_anom_idx if np.random.rand() < dev_prob else card_norm_idx)
        s_idx  = np.random.choice(stress_elev_idx if np.random.rand() < dev_prob else stress_norm_idx)
        sl_idx = np.random.choice(sleep_poor_idx if np.random.rand() < dev_prob else sleep_good_idx)

        X_seq[i, t, :] = np.concatenate([E_card[c_idx], E_stress[s_idx], E_sleep[sl_idx], z_ctx])

        c_p, s_p, r_p = P_card[c_idx], P_stress[s_idx], R_sleep[sl_idx]
        anom_val = 0.35 * c_p + 0.30 * s_p + 0.25 * (1.0 - r_p) + 0.10 * min(1.0, ctx_risk)
        if t > 0:
            anom_val = 0.70 * anom_val + 0.30 * Y_anom[i, t - 1]

        Y_anom[i, t] = np.clip(anom_val, 0.0, 1.0)
        Y_rec[i, t]  = np.clip(r_p - 0.15 * s_p, 0.0, 1.0)
        Y_str[i, t]  = np.clip(s_p, 0.0, 1.0)

# Train on first 480 trajectories, evaluate on 120 held-out test trajectories
train_ds = TensorDataset(
    torch.tensor(X_seq[:480]), torch.tensor(Y_anom[:480]),
    torch.tensor(Y_rec[:480]), torch.tensor(Y_str[:480])
)
loader = DataLoader(train_ds, batch_size=32, shuffle=True)

engine = TemporalBaselineEngine(128, 64).to(device)
optimizer = torch.optim.AdamW(engine.parameters(), lr=2e-3, weight_decay=1e-4)
criterion = nn.SmoothL1Loss()

engine.train()
for epoch in range(30):
    for bx, ba, br, bs in loader:
        bx, ba, br, bs = bx.to(device), ba.to(device), br.to(device), bs.to(device)
        optimizer.zero_grad()
        pa, pr, ps, _ = engine(bx)
        loss = criterion(pa, ba) + criterion(pr, br) + criterion(ps, bs)
        loss.backward()
        optimizer.step()

engine.eval()
with torch.no_grad():
    x_test_t = torch.tensor(X_seq[480:], device=device)
    pa_t, pr_t, ps_t, _ = engine(x_test_t)
    pa_np, pr_np, ps_np = pa_t.cpu().numpy(), pr_t.cpu().numpy(), ps_t.cpu().numpy()

mae_anom = np.mean(np.abs(pa_np - Y_anom[480:]))
mae_rec  = np.mean(np.abs(pr_np - Y_rec[480:]))
mae_str  = np.mean(np.abs(ps_np - Y_str[480:]))
r2_anom  = 1.0 - np.sum((pa_np - Y_anom[480:])**2) / np.sum((Y_anom[480:] - np.mean(Y_anom[480:]))**2)

fusion_weights_path = os.path.join(MODELS_DIR, "avia_temporal_fusion_engine.pt")
torch.save(engine.state_dict(), fusion_weights_path)

print("=========================================================================")
print(f"[+] Saved Trained Fusion + GRU Engine -> {fusion_weights_path}")
print(f"[+] Test Anomaly MAE: {mae_anom:.4f} (R² = {r2_anom:.4f})")
print(f"[+] Test Recovery MAE: {mae_rec:.4f} | Test Stress MAE: {mae_str:.4f}")
print("=========================================================================")

In [ ]:
import shap

# Initialize the 4 real-time encoders using your saved processed CSVs and weights
cardiac_rt = RealtimeCardiacInference(
    csv_path=os.path.join(BIDMC_DIR, "bidmc_features_sanitized_personal_baseline.csv"),
    model_weights_path=os.path.join(MODELS_DIR, "avia_cardiac_resp_encoder.pt")
)
stress_rt = RealtimeStressInference(
    csv_path=os.path.join(WESAD_DIR, "wesad_features_sanitized_personal_baseline.csv"),
    model_weights_path=os.path.join(MODELS_DIR, "avia_stress_encoder.pt")
)
sleep_rt = RealtimeSleepInference(
    csv_path=os.path.join(SLEEP_DIR, "sleep_edf_sc4001_epochs_features.csv"),
    model_weights_path=os.path.join(MODELS_DIR, "avia_sleep_encoder.pt")
)
nasa_ctx_rt = AVIANASAContextEncoder(cbc_csv_path=cbc_path, osd919_stats_csv_path=osd919_path)


def generate_astronaut_state_profile(
    crew_id: str = "C001",
    mission_timepoint: str = "R+1",
    # Current live vitals (from A.V.I.A. Pulse or Manual Simulator UI)
    hr_bpm: float = 76.0,
    hrv_rmssd_ms: float = 39.0,
    hrv_sdnn_ms: float = 45.0,
    resp_rate_bpm: float = 17.0,
    spo2_pct: float = 97.0,
    eda_tonic_us: float = 5.8,
    scr_peaks_min: float = 24.0,
    skin_temp_c: float = 34.3,
    sleep_duration_hr: float = 5.8,
    eeg_delta_rel: float = 0.68,
    eeg_theta_rel: float = 0.20,
    eeg_sigma_rel: float = 0.015,
    eeg_beta_rel: float = 0.030,
    eog_std_uv: float = 32.0,
    # Astronaut's Personal Historical Baseline (Section 9.2 of Documentation)
    base_hr_mean: float = 64.0, base_hr_std: float = 5.0,
    base_hrv_mean: float = 58.0, base_hrv_std: float = 10.0,
    base_sleep_mean: float = 7.4, base_sleep_std: float = 0.6,
    base_resp_mean: float = 14.0, base_resp_std: float = 2.0,
    base_eda_mean: float = 3.0, base_eda_std: float = 0.8
) -> dict:
    # 1. Encode each modality into its 32-dim representation
    c_out  = cardiac_rt.encode_from_manual_ui(
        hr_bpm=hr_bpm, rmssd_ms=hrv_rmssd_ms, sdnn_ms=hrv_sdnn_ms,
        resp_rate_bpm=resp_rate_bpm, spo2_pct=spo2_pct,
        baseline_hr=base_hr_mean, baseline_rmssd=base_hrv_mean,
        baseline_sdnn=base_hrv_mean, baseline_resp=base_resp_mean
    )
    st_out = stress_rt.encode_from_manual_ui(
        eda_tonic_us=eda_tonic_us, scr_peaks_per_min=scr_peaks_min,
        skin_temp_c=skin_temp_c, hr_bpm=hr_bpm, rmssd_ms=hrv_rmssd_ms,
        sdnn_ms=hrv_sdnn_ms, resp_rate_bpm=resp_rate_bpm,
        baseline_eda_tonic=base_eda_mean, baseline_hr=base_hr_mean,
        baseline_rmssd=base_hrv_mean, baseline_sdnn=base_hrv_mean
    )
    sl_out = sleep_rt.encode_from_manual_ui(
        delta_rel=eeg_delta_rel, theta_rel=eeg_theta_rel,
        sigma_rel=eeg_sigma_rel, beta_rel=eeg_beta_rel, eog_std_uv=eog_std_uv
    )
    cx_out = nasa_ctx_rt.get_context_embedding(crew_id=crew_id, mission_timepoint=mission_timepoint)

    # 2. Construct 72h (6-step) progressive trajectory from astronaut's baseline to current state
    base_c  = cardiac_rt.encode_from_manual_ui(hr_bpm=base_hr_mean, rmssd_ms=base_hrv_mean, sdnn_ms=base_hrv_mean, resp_rate_bpm=base_resp_mean)["e_cardiac"]
    base_st = stress_rt.encode_from_manual_ui(eda_tonic_us=base_eda_mean, hr_bpm=base_hr_mean, rmssd_ms=base_hrv_mean)["e_stress"]
    base_sl = sleep_rt.encode_from_manual_ui(delta_rel=0.90, theta_rel=0.06, beta_rel=0.005, eog_std_uv=15.0)["e_sleep"]
    base_cx = nasa_ctx_rt.get_context_embedding(crew_id=crew_id, mission_timepoint="L-3")["z_context"]

    vec_base = np.concatenate([base_c, base_st, base_sl, base_cx])
    vec_curr = np.concatenate([c_out["e_cardiac"], st_out["e_stress"], sl_out["e_sleep"], cx_out["z_context"]])

    # Interpolate 6 steps (Day 1 Normal -> Day 3 Current State)
    alphas = np.linspace(0.0, 1.0, 6)[:, None]
    seq_72h = (1.0 - alphas) * vec_base[None, :] + alphas * vec_curr[None, :]
    seq_tensor = torch.tensor(seq_72h[None, :, :], dtype=torch.float32, device=device)

    engine.eval()
    with torch.no_grad():
        anom_traj, rec_traj, str_traj, _ = engine(seq_tensor)
        anom_now = float(anom_traj[0, -1].cpu().item())
        rec_now  = float(rec_traj[0, -1].cpu().item())
        str_now  = float(str_traj[0, -1].cpu().item())

    # 3. Personal Baseline Z-Score Deviations (Section 9.2)
    z_scores = {
        "HR_z":    round((hr_bpm - base_hr_mean) / base_hr_std, 2),
        "HRV_z":   round((hrv_rmssd_ms - base_hrv_mean) / base_hrv_std, 2),
        "Sleep_z": round((sleep_duration_hr - base_sleep_mean) / base_sleep_std, 2),
        "Resp_z":  round((resp_rate_bpm - base_resp_mean) / base_resp_std, 2),
        "EDA_z":   round((eda_tonic_us - base_eda_mean) / base_eda_std, 2)
    }

    # 4. Counterfactual / Shapley Modality Attribution (Why did Anomaly Score change?)
    modality_slices = {
        "Cardiovascular & Resp": slice(0, 32),
        "Autonomic Stress (EDA/HRV)": slice(32, 64),
        "Sleep & Recovery": slice(64, 96),
        "Spaceflight Context (OSD-569/919)": slice(96, 128)
    }
    mod_contrib = {}
    with torch.no_grad():
        for m_name, sl in modality_slices.items():
            seq_cf = seq_tensor.clone()
            seq_cf[0, :, sl] = torch.tensor(vec_base[sl], dtype=torch.float32, device=device)
            anom_cf, _, _, _ = engine(seq_cf)
            mod_contrib[m_name] = max(0.005, anom_now - float(anom_cf[0, -1].cpu().item()))

    tot_c = sum(mod_contrib.values())
    mod_attribution_pct = {k: round((v / tot_c) * 100.0, 1) for k, v in mod_contrib.items()}

    # 5. Format Decision-Support States (Adhering strictly to Section 10.1 & Claims Guardrails)
    card_state  = "Elevated Deviation" if c_out["cardiac_anomaly_prob"] > 0.55 else ("Mild Deviation" if abs(z_scores["HR_z"]) > 1.8 else "Stable")
    resp_state  = "Elevated" if abs(z_scores["Resp_z"]) > 2.0 else "Stable"
    sleep_state = "Reduced" if (sleep_duration_hr < 6.2 or rec_now < 0.55) else "Optimal"
    str_state   = "Elevated" if (str_now > 0.50 or z_scores["EDA_z"] > 1.8) else "Baseline"
    anom_label  = "High" if anom_now >= 0.70 else ("Moderate" if anom_now >= 0.40 else "Low / Nominal")

    quality_flags = [c_out["data_quality_ok"], st_out["data_quality_ok"], sl_out["data_quality_ok"]]
    data_quality_pct = round(np.mean(quality_flags) * 100.0, 1)
    confidence_pct   = round(min(98.0, data_quality_pct * (0.82 + 0.15 * abs(anom_now - 0.5))), 1)

    # Generate human-readable explanation bullets
    reasons = []
    if z_scores["Sleep_z"] < -1.0:
        reasons.append(f"↓ sleep duration ({sleep_duration_hr}h vs {base_sleep_mean}h personal baseline, z={z_scores['Sleep_z']:+.1f})")
    if z_scores["EDA_z"] > 1.2 or str_state == "Elevated":
        reasons.append(f"↑ physiological stress indicators (EDA z={z_scores['EDA_z']:+.1f})")
    if z_scores["HRV_z"] < -1.0 or z_scores["HR_z"] > 1.2:
        reasons.append(f"↑ HRV/HR deviation from baseline (HRV {hrv_rmssd_ms}ms vs {base_hrv_mean}ms, z={z_scores['HRV_z']:+.1f})")
    if resp_state == "Stable":
        reasons.append(f"stable respiration ({resp_rate_bpm} bpm)")
    if mission_timepoint in ["R+1", "R+45"]:
        reasons.append(f"spaceflight context prior ({crew_id} {mission_timepoint}: NLR shift {cx_out['human_nlr_shift_pct']:+.1f}%)")

    profile = {
        "Cardiovascular": card_state,
        "Respiratory": resp_state,
        "Sleep / Recovery": sleep_state,
        "Stress": str_state,
        "Anomaly_Level": anom_label,
        "Anomaly_Score": round(anom_now, 3),
        "Recovery_Score": round(rec_now, 3),
        "Stress_Score": round(str_now, 3),
        "Confidence_Pct": confidence_pct,
        "Data_Quality_Pct": data_quality_pct,
        "Personal_Baseline_Z_Scores": z_scores,
        "Modality_Attributions_Pct": mod_attribution_pct,
        "Trajectory_72h_Anomaly": np.round(anom_traj[0].cpu().numpy(), 3).tolist(),
        "Why_Explanation": " | ".join(reasons)
    }
    return profile

# Test with the exact scenario from Section 9.2 & 10.1 of your documentation!
demo_profile = generate_astronaut_state_profile()

print("==================================================================")
print("                ASTRONAUT PHYSIOLOGICAL PROFILE                   ")
print("        (Prototype Decision-Support — Non-Diagnostic)             ")
print("==================================================================")
print(f"  Cardiovascular   : {demo_profile['Cardiovascular']}")
print(f"  Respiratory      : {demo_profile['Respiratory']}")
print(f"  Sleep / Recovery : {demo_profile['Sleep / Recovery']} (Recovery Index: {demo_profile['Recovery_Score']:.2f})")
print(f"  Stress           : {demo_profile['Stress']} (Stress Pattern Prob: {demo_profile['Stress_Score']:.2f})")
print(f"  Anomaly          : {demo_profile['Anomaly_Level']} (Score: {demo_profile['Anomaly_Score']:.3f})")
print(f"  Confidence       : {demo_profile['Confidence_Pct']}% (Signal Quality: {demo_profile['Data_Quality_Pct']}%)")
print("------------------------------------------------------------------")
print(f"  Why? {demo_profile['Why_Explanation']}")
print("------------------------------------------------------------------")
print("  Modality Contributions to Anomaly Score:")
for mod_k, mod_v in demo_profile["Modality_Attributions_Pct"].items():
    print(f"    • {mod_k:34s}: {mod_v:5.1f}%")
print("==================================================================")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
time_hours = [0, 12, 24, 36, 48, 72]

stable_mask = np.arange(len(pa_np)) % 2 == 0
dev_mask    = np.arange(len(pa_np)) % 2 == 1

# Panel A: 72-Hour Trajectory Plot
axes[0].plot(time_hours, pa_np[dev_mask].mean(axis=0), marker="o", color="#d95f02", linewidth=2.5, label="Progressive Deviation Trajectory (Anomaly Score)")
axes[0].fill_between(
    time_hours,
    pa_np[dev_mask].mean(axis=0) - pa_np[dev_mask].std(axis=0),
    pa_np[dev_mask].mean(axis=0) + pa_np[dev_mask].std(axis=0),
    color="#d95f02", alpha=0.2
)
axes[0].plot(time_hours, pr_np[dev_mask].mean(axis=0), marker="s", linestyle="--", color="#7570b3", linewidth=2.0, label="Sleep / Recovery Trajectory (Deviating Crew)")
axes[0].plot(time_hours, pa_np[stable_mask].mean(axis=0), marker="^", color="#1b9e77", linewidth=2.2, label="Nominal Baseline Crew (Anomaly Score)")
axes[0].axhline(0.60, color="red", linestyle=":", linewidth=1.2, label="Decision-Support Alert Threshold (0.60)")
axes[0].set_title("A. 72-Hour Temporal State Trajectory (GRU Engine)\nProgressive Baseline Deviation vs. Nominal Crew", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Mission Elapsed Monitoring Time (Hours)")
axes[0].set_ylabel("Physiological State Indicator (0.0 to 1.0)")
axes[0].set_ylim(0.0, 1.05)
axes[0].legend(loc="center left", fontsize=8.2)
axes[0].grid(True, linestyle=":", alpha=0.6)

# Panel B: Modality Attribution Bar Chart
names = list(demo_profile["Modality_Attributions_Pct"].keys())
vals  = list(demo_profile["Modality_Attributions_Pct"].values())
colors = ["#1b9e77", "#d95f02", "#3182bd", "#756bb1"]

bars = axes[1].barh(names, vals, color=colors, edgecolor="black", height=0.55)
for bar, v in zip(bars, vals):
    axes[1].text(v + 0.8, bar.get_y() + bar.get_height()/2, f"{v:.1f}%", va="center", fontsize=9.5, fontweight="bold")

axes[1].set_xlim(0, max(vals) + 12)
axes[1].set_title("B. Multimodal Attribution (Why Anomaly Elevated)\nCounterfactual / Shapley Modality Contribution (%)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Relative Contribution to Anomaly Score (%)")
axes[1].grid(axis="x", linestyle=":", alpha=0.6)

plt.suptitle("A.V.I.A. Step 4: Multimodal Intermediate Fusion & 72h Temporal Baseline Validation", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()

out_fig = os.path.join(FIGURES_DIR, "avia_step4_multimodal_fusion_validation.png")
plt.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"[+] Saved Step 4 Multimodal Validation Figure -> {out_fig}")